In [4]:
import pandas as pd
import sqlite3

Parsing SQLite tables to dataframes

In [32]:
dbconn = sqlite3.connect('location.db')
tables = list(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", dbconn)['name'])

query_1 = """
    SELECT DISTINCT timestamp, stopId, [trip.tripId], [trip.directionId], [trip.startDate] FROM
    """
    
query_2 = """
    WHERE [trip.routeId] = 6641 AND [trip.directionId] = 1 AND (stopId = 11606 OR stopId = 12483)
    GROUP BY [trip.tripId]
    
"""

location_dfs = []

for table in tables:
    table_name = "\'" + table + "\'"
    query = query_1 + table_name + query_2
    
    location_dfs.append(pd.read_sql_query(query, dbconn))

In [31]:
location_dfs[0].head()

,timestamp,stopId,trip.tripId,trip.directionId,trip.startDate
0,1776351215,12483,14898246,1,20260416
1,1776351473,12483,14898247,1,20260416
2,1776352160,12483,14898248,1,20260416
3,1776351683,12483,14898249,1,20260416
4,1776352215,12483,14898250,1,20260416


Cleaning & transforming timestamps into relevant delta times

In [29]:
delta_time_dfs = []
for df in location_dfs:
    maxed_df = df.groupby(["stopId", "trip.tripId"]).max()
    # maxed_df = df[df['timestamp'] == df['timestamp'].max()]
    maxed_df["timestamp"] = pd.to_datetime(maxed_df["timestamp"], unit = 's', errors = 'coerce')
    maxed_df.reset_index()
    maxed_df.sort_values(["trip.tripId"])

    delta_time_dfs.append(maxed_df)
    
delta_time_dfs[0]

timestamp  trip.directionId  trip.startDate
stopId trip.tripId                                                      
11606  14898260    2026-04-16 15:01:15                 1        20260416
       14898261    2026-04-16 15:04:11                 1        20260416
       14898262    2026-04-16 15:08:08                 1        20260416
       14898263    2026-04-16 15:11:56                 1        20260416
       14898264    2026-04-16 15:14:41                 1        20260416
...                                ...               ...             ...
12483  14898310    2026-04-16 19:01:49                 1        20260416
       14898311    2026-04-16 19:06:15                 1        20260416
       14898312    2026-04-16 19:15:40                 1        20260416
       14898313    2026-04-16 19:18:44                 1        20260416
       14898314    2026-04-16 19:22:42                 1        20260416

[103 rows x 3 columns]